# **SNU AI Challenge - CLIP + MSE Feature Extractor**

This notebook extracts semantic (CLIP) and visual (MSE) features for all images in the `train` and `test` splits.
- **Purpose**: Creates enriched CSV files (`train_with_features.csv`, `test_with_features.csv`) containing scene transitions and camera dynamics to help VLM sequence reasoning.
- **Prerequisites**: Make sure to select your **GPU kernel** (which has PyTorch with CUDA installed) at the top-right of VS Code.

In [ ]:
import os
import sys
import ast
import ssl
import numpy as np
import pandas as pd
from PIL import Image
from tqdm.notebook import tqdm
from itertools import combinations

# Bypass SSL verification for downloads
ssl._create_default_https_context = ssl._create_unverified_context
os.environ["KMP_DUPLICATE_LIB_OK"] = "TRUE"

import torch
print("PyTorch Version:", torch.__version__)
print("CUDA Available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU Device Name:", torch.cuda.get_device_name(0))

## **1. Paths & Thresholds Setup**

In [ ]:
DATA_DIR = "./snuaichallenge_data"
TRAIN_CSV = os.path.join(DATA_DIR, "train.csv")
TEST_CSV = os.path.join(DATA_DIR, "test.csv")
TRAIN_IMG_DIR = os.path.join(DATA_DIR, "train")
TEST_IMG_DIR = os.path.join(DATA_DIR, "test")

OUTPUT_TRAIN_CSV = os.path.join(DATA_DIR, "train_with_features.csv")
OUTPUT_TEST_CSV = os.path.join(DATA_DIR, "test_with_features.csv")
EMBEDDING_CACHE_PATH = os.path.join(DATA_DIR, "clip_embeddings_cache.pt")

MSE_THRESHOLD = 1200
CLIP_THRESHOLD = 0.20

## **2. Helper Functions**

In [ ]:
def compute_image_mse(img_path1, img_path2):
    try:
        with Image.open(img_path1).convert('L') as img1, Image.open(img_path2).convert('L') as img2:
            img1_r = img1.resize((64, 64), Image.Resampling.BILINEAR)
            img2_r = img2.resize((64, 64), Image.Resampling.BILINEAR)
            arr1 = np.array(img1_r, dtype=np.float32)
            arr2 = np.array(img2_r, dtype=np.float32)
            return float(np.mean((arr1 - arr2) ** 2))
    except Exception:
        return 0.0

def map_similar_pairs_to_cuts(similar_pairs):
    if similar_pairs >= 5:
        return 0
    elif 2 <= similar_pairs <= 4:
        return 1
    elif similar_pairs == 1:
        return 2
    else:
        return 3

## **3. Load CLIP Model**

In [ ]:
device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Using device: {device}")

clip_model, clip_preprocess = torch.hub.load("openai/CLIP", "ViT_B_32", trust_repo=True)
clip_model = clip_model.to(device).eval()
print("CLIP loaded successfully.")

## **4. Collect Unique Images**

In [ ]:
train_df = pd.read_csv(TRAIN_CSV)
test_df = pd.read_csv(TEST_CSV)

image_paths = []
for _, row in train_df.iterrows():
    sample_id = str(row['Id'])
    for f in [row['Input_1'], row['Input_2'], row['Input_3'], row['Input_4']]:
        image_paths.append(os.path.join(TRAIN_IMG_DIR, sample_id, f))
        
for _, row in test_df.iterrows():
    sample_id = str(row['Id'])
    for f in [row['Input_1'], row['Input_2'], row['Input_3'], row['Input_4']]:
        image_paths.append(os.path.join(TEST_IMG_DIR, sample_id, f))
        
unique_paths = list(set(image_paths))
print(f"Total unique images: {len(unique_paths):,}")

## **5. Extract Embeddings (in Batches)**

In [ ]:
embedding_cache = {}
if os.path.exists(EMBEDDING_CACHE_PATH):
    print(f"Loading cached embeddings from {EMBEDDING_CACHE_PATH}...")
    embedding_cache = torch.load(EMBEDDING_CACHE_PATH, map_location='cpu')
    print(f"Loaded {len(embedding_cache):,} embeddings.")

missing_paths = [p for p in unique_paths if p not in embedding_cache]

if len(missing_paths) > 0:
    print(f"Extracting {len(missing_paths):,} missing embeddings...")
    batch_size = 128
    
    for i in tqdm(range(0, len(missing_paths), batch_size), desc="CLIP Inference"):
        batch_paths = missing_paths[i:i+batch_size]
        batch_imgs = []
        valid_paths = []
        
        for p in batch_paths:
            if os.path.exists(p):
                try:
                    img = Image.open(p).convert("RGB")
                    batch_imgs.append(clip_preprocess(img))
                    valid_paths.append(p)
                except Exception as e:
                    print(f"Error opening {p}: {e}")
                    
        if not batch_imgs:
            continue
            
        img_tensor = torch.stack(batch_imgs).to(device)
        
        with torch.no_grad():
            features = clip_model.encode_image(img_tensor)
            features = features / features.norm(p=2, dim=-1, keepdim=True)
            features_cpu = features.cpu().numpy()
            
        for path, feat in zip(valid_paths, features_cpu):
            embedding_cache[path] = feat
            
    # Save cache
    torch.save(embedding_cache, EMBEDDING_CACHE_PATH)
    print("Embeddings extraction completed and cached.")
else:
    print("All embeddings already exist in cache!")

## **6. Process Features and Save Enriched CSVs**

In [ ]:
def process_dataset(df, is_train=True):
    img_dir = TRAIN_IMG_DIR if is_train else TEST_IMG_DIR
    processed_rows = []
    
    for idx, row in tqdm(df.iterrows(), total=len(df), desc=f"Processing {'Train' if is_train else 'Test'}"):
        sample_id = str(row['Id'])
        shuffled_files = [row['Input_1'], row['Input_2'], row['Input_3'], row['Input_4']]
        img_paths = [os.path.join(img_dir, sample_id, f) for f in shuffled_files]
        
        pairs = list(combinations(range(4), 2))
        mse_list = []
        clip_dist_list = []
        
        for p1, p2 in pairs:
            mse_val = compute_image_mse(img_paths[p1], img_paths[p2])
            mse_list.append(mse_val)
            
            feat1 = embedding_cache.get(img_paths[p1])
            feat2 = embedding_cache.get(img_paths[p2])
            if feat1 is not None and feat2 is not None:
                cos_sim = float(np.dot(feat1, feat2))
                clip_dist = 1.0 - cos_sim
            else:
                clip_dist = 0.0
            clip_dist_list.append(clip_dist)
            
        similar_clip_pairs = sum([1 for c in clip_dist_list if c < CLIP_THRESHOLD])
        similar_mse_pairs = sum([1 for m in mse_list if m < MSE_THRESHOLD])
        predicted_cuts = map_similar_pairs_to_cuts(similar_clip_pairs)
        
        feat_dict = {
            'Id': row['Id'],
            'similar_clip_pairs': similar_clip_pairs,
            'similar_mse_pairs': similar_mse_pairs,
            'predicted_cuts': predicted_cuts,
            'min_clip_dist': float(np.min(clip_dist_list)),
            'max_clip_dist': float(np.max(clip_dist_list)),
            'mean_clip_dist': float(np.mean(clip_dist_list)),
            'std_clip_dist': float(np.std(clip_dist_list)),
            'min_mse': float(np.min(mse_list)),
            'max_mse': float(np.max(mse_list)),
            'mean_mse': float(np.mean(mse_list)),
            'std_mse': float(np.std(mse_list))
        }
        
        if is_train and 'Answer' in row:
            ans = ast.literal_eval(row['Answer'])
            ordered_files = [None] * 4
            for f_idx, pos in enumerate(ans):
                ordered_files[pos - 1] = shuffled_files[f_idx]
            ordered_paths = [os.path.join(img_dir, sample_id, f) for f in ordered_files]
            
            seq_mses = []
            seq_clips = []
            for i in range(3):
                seq_mses.append(compute_image_mse(ordered_paths[i], ordered_paths[i+1]))
                feat1 = embedding_cache.get(ordered_paths[i])
                feat2 = embedding_cache.get(ordered_paths[i+1])
                if feat1 is not None and feat2 is not None:
                    cos_sim = float(np.dot(feat1, feat2))
                    seq_clips.append(1.0 - cos_sim)
                else:
                    seq_clips.append(0.0)
                    
            actual_semantic_cuts = sum([1 for c in seq_clips if c >= CLIP_THRESHOLD])
            actual_visual_cuts = sum([1 for m in seq_mses if m >= MSE_THRESHOLD])
            
            feat_dict.update({
                'actual_semantic_cuts': actual_semantic_cuts,
                'actual_visual_cuts': actual_visual_cuts,
                'seq_mse_12': seq_mses[0],
                'seq_mse_23': seq_mses[1],
                'seq_mse_34': seq_mses[2],
                'seq_clip_12': seq_clips[0],
                'seq_clip_23': seq_clips[1],
                'seq_clip_34': seq_clips[2]
            })
            
        processed_rows.append(feat_dict)
        
    feat_df = pd.DataFrame(processed_rows)
    return pd.merge(df, feat_df, on='Id', how='left')

enriched_train_df = process_dataset(train_df, is_train=True)
enriched_test_df = process_dataset(test_df, is_train=False)

enriched_train_df.to_csv(OUTPUT_TRAIN_CSV, index=False)
enriched_test_df.to_csv(OUTPUT_TEST_CSV, index=False)

print("Finished successfully!")
print(f"Train file saved to: {OUTPUT_TRAIN_CSV} (Shape: {enriched_train_df.shape})")
print(f"Test file saved to: {OUTPUT_TEST_CSV} (Shape: {enriched_test_df.shape})")